# 01 — Data Exploration

Explore the CREMA-D dataset: class distribution, and a side-by-side comparison of Mel-spectrogram vs MFCC features for the same audio file.

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from project root

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from feature_extraction import extract_mel, extract_mfcc

## Dataset Statistics

In [ ]:
df = pd.read_csv('../crema_metadata.csv')

print(f"Total samples : {len(df)}")
print(f"Unique actors : {df['actor_id'].nunique()}")
print(f"Emotions      : {sorted(df['emotion_name'].unique())}")
print()
print(df.groupby('emotion_name')['emotion_id'].count().rename('count').to_string())

In [ ]:
emotion_counts = df.groupby('emotion_name')['emotion_id'].count().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
bars = plt.bar(emotion_counts.index, emotion_counts.values, color='steelblue', edgecolor='white')
plt.title('CREMA-D — Samples per Emotion', fontsize=14)
plt.xlabel('Emotion')
plt.ylabel('Number of samples')
plt.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, emotion_counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, str(val),
             ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## Mel-spectrogram vs MFCC — Side-by-Side

Both representations are extracted from the same audio file. The model input shape is **(1, 40, 200)** for both — 40 frequency bins × 200 time frames.

In [ ]:
# Pick one sample per emotion for comparison
emotions = sorted(df['emotion_name'].unique())

fig, axes = plt.subplots(len(emotions), 2, figsize=(14, 3 * len(emotions)))

for i, emotion in enumerate(emotions):
    row = df[df['emotion_name'] == emotion].iloc[0]
    wav_path = row['file_path']

    mel  = extract_mel(f'../{wav_path}')
    mfcc = extract_mfcc(f'../{wav_path}')

    axes[i, 0].imshow(mel,  aspect='auto', origin='lower', cmap='viridis')
    axes[i, 0].set_title(f'{emotion} — Mel-spectrogram', fontsize=11)
    axes[i, 0].set_xlabel('Time frame')
    axes[i, 0].set_ylabel('Mel bin')

    axes[i, 1].imshow(mfcc, aspect='auto', origin='lower', cmap='plasma')
    axes[i, 1].set_title(f'{emotion} — MFCC', fontsize=11)
    axes[i, 1].set_xlabel('Time frame')
    axes[i, 1].set_ylabel('MFCC coefficient')

plt.suptitle('Mel-spectrogram vs MFCC — one sample per emotion', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()